# English -> Urdu fine-tuning of NLLB

In [1]:
# BLEU and chrF++ are always computed. These two are the extra metrics:
RUN_METEOR = True     # lexical metric, cheap (see the note in step 10 about Urdu)
RUN_COMET = True      # neural semantic metric; downloads a COMET model (~2.3 GB) from Hugging Face
COMET_MODEL = "Unbabel/wmt22-comet-da"   # lighter/faster option: "Unbabel/eamt22-cometinho-da"
COMET_BS = 16         # lower to 8 if COMET runs out of GPU memory

import sys, subprocess, importlib, importlib.util
from importlib.metadata import version as _pkg_version, PackageNotFoundError

# Kaggle wipes pip installs on every "Restart session" and already has numpy/torch loaded when this cell runs.
# So every install below is done with the key packages FROZEN (constraints file): pip can never change numpy /
# pandas / pyarrow / torch / transformers ... it either fits with them or fails. No restart is ever needed.
_PINNED = ["numpy", "pandas", "pyarrow", "scipy", "scikit-learn", "torch", "transformers", "tokenizers",
           "datasets", "huggingface-hub", "protobuf", "sentencepiece", "safetensors", "accelerate", "nltk"]
_pins = []
for _p in _PINNED:
    try:
        _pins.append(f"{_p}=={_pkg_version(_p)}")
    except PackageNotFoundError:
        pass
CONSTRAINTS = "/tmp/constraints.txt"
with open(CONSTRAINTS, "w") as _f:
    _f.write("\n".join(_pins) + "\n")
_PIP = [sys.executable, "-m", "pip", "install", "-q", "-c", CONSTRAINTS]


def _ensure(module, pip_name=None):
    """pip-install a package only if it is missing (frozen-constraints install, then make it importable)."""
    if importlib.util.find_spec(module) is None:
        print(f"installing {pip_name or module} ...")
        subprocess.check_call(_PIP + [pip_name or module])
        importlib.invalidate_caches()


for _m in ["transformers", "datasets", "sacrebleu", "sentencepiece", "accelerate", "nltk"]:
    _ensure(_m)

# unbabel-comet pins old versions (numpy<2, transformers<5 ...). A pip WARNING about that is harmless: it is installed
# WITHOUT its dependency pins, plus the few small packages it needs.
if RUN_COMET and importlib.util.find_spec("comet") is None:
    try:
        print("installing unbabel-comet ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "unbabel-comet"])
        subprocess.check_call(_PIP + ["entmax", "jsonargparse", "pytorch-lightning", "torchmetrics"])
        importlib.invalidate_caches()
    except Exception as e:  # noqa
        print("COMET install failed -> COMET disabled (everything else still runs):", repr(e))
        RUN_COMET = False

if RUN_COMET:    # fail early (now) instead of after hours of training: import + score ONE sentence on the CPU
    try:
        from comet import download_model, load_from_checkpoint
        _cm = load_from_checkpoint(download_model(COMET_MODEL))      # also pre-downloads the model (cached for step 14)
        _t = _cm.predict([{"src": "Hello", "mt": "ہیلو", "ref": "ہیلو"}], batch_size=1, gpus=0, progress_bar=False)
        print("COMET smoke test OK, score:", round(float(_t.system_score), 4))
        del _cm, _t
    except Exception as e:  # noqa
        print("COMET does not work in this environment -> COMET disabled (everything else still runs):", repr(e))
        RUN_COMET = False
# LaBSE filter (USE_LABSE_FILTER=True below) also needs:  !pip install -q sentence-transformers

installing sacrebleu ...
installing unbabel-comet ...


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unbabel-comet 2.2.7 requires huggingface-hub<1.0,>=0.19.3, but you have huggingface-hub 1.29.0 which is incompatible.
unbabel-comet 2.2.7 requires jsonargparse==3.13.1, but you have jsonargparse 4.53.0 which is incompatible.
unbabel-comet 2.2.7 requires numpy<2.0.0,>=1.20.0, but you have numpy 2.1.3 which is incompatible.
unbabel-comet 2.2.7 requires protobuf<5.0.0,>=4.24.4, but you have protobuf 5.29.5 which is incompatible.
unbabel-comet 2.2.7 requires torchmetrics<0.11.0,>=0.10.2, but you have torchmetrics 1.9.0 which is incompatible.
unbabel-comet 2.2.7 requires transformers<5.0,>=4.17, but you have transformers 5.16.1 which is incompatible.
HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
HTTP Request: GET https://huggingface.co/api/models/Unbabel/wmt22-comet-da/revision/main "HT

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

HTTP Request: HEAD https://huggingface.co/Unbabel/wmt22-comet-da/resolve/2760a223ac957f30acfb18c8aa649b01cf1d75f2/README.md "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Unbabel/wmt22-comet-da/2760a223ac957f30acfb18c8aa649b01cf1d75f2/README.md "HTTP/1.1 200 OK"
HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Unbabel/wmt22-comet-da/2760a223ac957f30acfb18c8aa649b01cf1d75f2/README.md "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/Unbabel/wmt22-comet-da/resolve/2760a223ac957f30acfb18c8aa649b01cf1d75f2/hparams.yaml "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/Unbabel/wmt22-comet-da/resolve/2760a223ac957f30acfb18c8aa649b01cf1d75f2/.gitattributes "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Unbabel/wmt22-comet-da/2760a223ac957f30acfb18c8aa649b01cf1d75f2/hparams.yaml "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

... (4 outputs left out)

HTTP Request: HEAD https://huggingface.co/xlm-roberta-large/resolve/main/added_tokens.json "HTTP/1.1 404 Not Found"
HTTP Request: HEAD https://huggingface.co/xlm-roberta-large/resolve/main/special_tokens_map.json "HTTP/1.1 404 Not Found"
HTTP Request: HEAD https://huggingface.co/xlm-roberta-large/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"
HTTP Request: GET https://huggingface.co/api/models/xlm-roberta-large "HTTP/1.1 307 Temporary Redirect"
HTTP Request: GET https://huggingface.co/api/models/FacebookAI/xlm-roberta-large "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/xlm-roberta-large/resolve/main/config.json "HTTP/1.1 200 OK"
HTTP Request: GET https://huggingface.co/xlm-roberta-large/resolve/main/config.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

Encoder model frozen.
/usr/local/lib/python3.13/dist-packages/pytorch_lightning/core/saving.py:216: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
GPU available: True (cuda), used: False
TPU available: False, using: 0 TPU cores
/usr/local/lib/python3.13/dist-packages/pytorch_lightning/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
/usr/local/lib/python3.13/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, Leaf

COMET does not work in this environment -> COMET disabled (everything else still runs): ValueError('not enough values to unpack (expected 3, got 2)')


In [2]:
import os, gc, re, json, html, inspect, unicodedata, warnings
from collections import Counter

# reduce GPU memory fragmentation (must be set BEFORE torch touches the GPU)
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sacrebleu.metrics import BLEU, CHRF
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq,
    Seq2SeqTrainer, Seq2SeqTrainingArguments, EarlyStoppingCallback, set_seed,
)

warnings.filterwarnings("ignore")

# ---- files: name -> domain tag (used for stratified splits and per-domain scores)
DATASET_DIRS = {   # domain -> Kaggle folder
    "general":  "/kaggle/input/datasets/atifkhan12/english-urdu",
    "parallel": "/kaggle/input/datasets/atifkhan12/parallel-corpus",
    "quran":    "/kaggle/input/datasets/atifkhan12/quran-df",
}
EXPECTED_NAMES = {"general": "english_urdu", "parallel": "parallel-corpus", "quran": "quran"}


def _pick_csv(folder, expected):
    if not os.path.isdir(folder):
        raise FileNotFoundError(f"{folder} not found. Add this dataset to the notebook (Add Input).")
    cands = [os.path.join(dp, fn) for dp, _, fns in os.walk(folder) for fn in fns
             if fn.lower().endswith((".csv", ".tsv", ".txt"))]
    if not cands:
        raise FileNotFoundError(f"No CSV file inside {folder}")
    norm = lambda s: re.sub(r"[^a-z0-9]", "", s.lower())
    close = [c for c in cands if norm(expected) in norm(os.path.basename(c))]
    return max(close or cands, key=os.path.getsize)     # best name match, else the largest file


FILES, DATA_PATHS, SEARCH_DIRS = {}, {}, []   # same variables the rest of the script uses
for _domain, _folder in DATASET_DIRS.items():
    _path = _pick_csv(_folder, EXPECTED_NAMES[_domain])
    _name = os.path.basename(_path)
    if _name in FILES:
        raise ValueError(f"Two datasets contain a file called {_name}; rename one or set FILES manually.")
    FILES[_name] = _domain
    DATA_PATHS[_name] = _path
    SEARCH_DIRS.append(os.path.dirname(_path))
    print(f"{_domain:9s} -> {_path}")

# If auto-detection picks the wrong columns, set them here:
# COLUMN_OVERRIDES = {"quran.csv": ("English_Column_Name", "Urdu_Column_Name")}
COLUMN_OVERRIDES = {}

# ---- model / languages
MODEL_NAME = "facebook/nllb-200-distilled-600M"   # try ...-1.3B if you have the GPU memory
SRC_LANG, TGT_LANG = "eng_Latn", "urd_Arab"

# ---- preprocessing
MAX_LEN = 200                 # max tokens per side; longer pairs are DROPPED (not truncated)
MAX_WORDS = 200
MIN_SCRIPT_RATIO = 0.6        # share of letters that must be in the expected script
WORD_RATIO_RANGE = (1 / 3, 3.0)
STRIP_URDU_DIACRITICS = True
USE_LABSE_FILTER = False      # needs sentence-transformers + internet to download LaBSE
LABSE_THRESHOLD = 0.60

# ---- splits
VAL_MAX, TEST_MAX = 3000, 3000
VAL_EVAL_SUBSET = 800         # validation rows used for per-epoch eval during training (speed)
MAX_TRAIN_ROWS = None         # e.g. 50000 for a quick smoke test

# ---- training
SEED = 42
EPOCHS = 5
LR = 1e-4                     # with Adafactor; use ~3e-5 to 5e-5 if you switch to AdamW
TRAIN_BS, EVAL_BS, GRAD_ACC = 4, 8, 4           # per GPU; effective batch = 4 x n_gpus x 4 (32 on 2 GPUs)
                                                # CUDA out of memory? -> TRAIN_BS=2, GRAD_ACC=8
FREEZE_EMBEDDINGS = True      # saves lots of memory (256k-vocab embeddings are tied to lm_head)
OPTIM = "adafactor"           # "adafactor" (low memory) or "adamw_torch"
LABEL_SMOOTHING = 0.0         # 0.1 builds a full log-softmax over the 256k vocab and often runs out of memory on T4s
RESUME_FROM = None            # path to a checkpoint to resume

# ---- evaluation
RUN_BASELINE = True           # zero-shot NLLB scored on the same test set
NUM_BEAMS = 5
GEN_BATCH = 16               # lower to 8 if generation runs out of memory
# RUN_METEOR / RUN_COMET / COMET_MODEL / COMET_BS are set in step 0

OUT_DIR = "/kaggle/working/nllb_en_ur" if os.path.isdir("/kaggle/working") else "./nllb_en_ur"
os.makedirs(OUT_DIR, exist_ok=True)

set_seed(SEED)
USE_CUDA = torch.cuda.is_available()
print("CUDA:", USE_CUDA, "|", torch.cuda.get_device_name(0) if USE_CUDA else "CPU (training will be very slow)")

TensorFlow version 2.20.0 available.
JAX version 0.11.1 available.


general   -> /kaggle/input/datasets/atifkhan12/english-urdu/english_urdu.csv
parallel  -> /kaggle/input/datasets/atifkhan12/parallel-corpus/parallel-corpus.csv
quran     -> /kaggle/input/datasets/atifkhan12/quran-df/quran.csv
CUDA: True | Tesla T4


In [3]:
_ZW = re.compile(r"[\u200b\u200c\u200d\u200e\u200f\u202a-\u202e\u2066-\u2069\ufeff]")
_CTRL = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]")
_HTML = re.compile(r"<[^>]+>")
_URL = re.compile(r"https?://\S+|www\.\S+")
_FOOT = re.compile(r"\[\d+\]")
_WS = re.compile(r"\s+")
# harakat, superscript alef, Quranic annotation marks (U+06D6-U+06ED), tatweel
_DIAC = re.compile(r"[\u064B-\u065F\u0670\u06D6-\u06ED\u0640]")

EN_PUNCT = str.maketrans({"\u2018": "'", "\u2019": "'", "\u201c": '"', "\u201d": '"',
                          "\u2013": "-", "\u2014": "-", "\u00a0": " "})

# Arabic-variant letters -> Urdu letters; Arabic-Indic digits -> ASCII
_ur_map = {"ي": "ی", "ى": "ی", "ك": "ک", "ه": "ہ", "ۀ": "ۂ"}
_ur_map.update({chr(0x0660 + i): str(i) for i in range(10)})
_ur_map.update({chr(0x06F0 + i): str(i) for i in range(10)})
UR_MAP = str.maketrans(_ur_map)


def clean_en(t):
    t = html.unescape(str(t))
    t = _HTML.sub(" ", t)
    t = _URL.sub(" ", t)
    t = unicodedata.normalize("NFKC", t).translate(EN_PUNCT)
    t = _ZW.sub("", t)
    t = _CTRL.sub(" ", t)
    t = _FOOT.sub(" ", t)
    return _WS.sub(" ", t).strip()


def clean_ur(t):
    t = html.unescape(str(t))
    t = _HTML.sub(" ", t)
    t = _URL.sub(" ", t)
    t = unicodedata.normalize("NFKC", t)
    t = _ZW.sub("", t)
    t = _CTRL.sub(" ", t)
    t = _FOOT.sub(" ", t)
    if STRIP_URDU_DIACRITICS:
        t = _DIAC.sub("", t)
    t = t.translate(UR_MAP)
    return _WS.sub(" ", t).strip()


def script_ratio(t, kind):
    letters = [c for c in t if c.isalpha()]
    if not letters:
        return 0.0
    if kind == "ur":
        n = sum(("\u0600" <= c <= "\u06ff") or ("\u0750" <= c <= "\u077f") or ("\ufb50" <= c <= "\ufeff")
                for c in letters)
    else:
        n = sum(c.isascii() or ("\u00c0" <= c <= "\u024f") for c in letters)
    return n / len(letters)

In [4]:
def find_file(name):
    """Resolve the CSV for a dataset: exact file name first, else the only/best-matching CSV in the folder."""
    p = DATA_PATHS[name]
    if os.path.isfile(p):
        return p
    if not os.path.isdir(p):
        raise FileNotFoundError(f"{p} does not exist. Check the dataset is added to this notebook.")
    csvs = []
    for dp, _, fns in os.walk(p):
        for fn in fns:
            if fn.lower().endswith((".csv", ".tsv", ".txt")):
                csvs.append(os.path.join(dp, fn))
    print(f"  files in {p}: {[os.path.basename(c) for c in csvs]}")
    if not csvs:
        raise FileNotFoundError(f"No CSV found in {p}")
    exact = [c for c in csvs if os.path.basename(c).lower() == name.lower()]
    if exact:
        return exact[0]
    if len(csvs) == 1:
        return csvs[0]
    stem = os.path.splitext(name)[0].lower().replace("-", "").replace("_", "")
    close = [c for c in csvs if stem in os.path.basename(c).lower().replace("-", "").replace("_", "")]
    if close:
        return close[0]
    raise FileNotFoundError(f"Several CSVs in {p} and none match '{name}': set DATA_PATHS['{name}'] to the exact file.")


def read_csv_robust(path):
    kw = dict(dtype=str, keep_default_na=False, on_bad_lines="skip")
    last, df, used = None, None, None
    for enc in ["utf-8-sig", "utf-8", "utf-16", "cp1256", "latin-1"]:
        try:
            df = pd.read_csv(path, encoding=enc, **kw)
            used = enc
            break
        except Exception as e:  # noqa
            last = e
    if df is None:
        raise last
    # headerless file? (a "header" that looks like data)
    if any(len(str(c)) > 60 or re.search(r"[\u0600-\u06ff]", str(c)) for c in df.columns):
        df = pd.read_csv(path, encoding=used, header=None, **kw)
        df.columns = [f"col{i}" for i in range(df.shape[1])]
    return df


UR_WORDS = {"کے", "کی", "ہے", "اور", "میں", "کو", "سے", "کہ", "ہیں", "نے", "کا", "پر"}
EN_WORDS = {"the", "of", "and", "to", "in", "is", "a", "that", "for", "with", "he", "you", "was", "it"}


def _urdu_score(s):
    s = s.astype(str).head(500)
    if not len(s):
        return 0.0
    return float(s.apply(lambda x: any(w in UR_WORDS for w in x.split())).mean())


def _english_score(s):
    s = s.astype(str).head(500)
    if not len(s):
        return 0.0
    hit = s.apply(lambda x: any(w in EN_WORDS for w in x.lower().split())).mean()
    asc = s.apply(lambda x: script_ratio(x, "en")).mean()
    return float(hit * asc)


def detect_columns(df, fname):
    if fname in COLUMN_OVERRIDES:
        return COLUMN_OVERRIDES[fname]
    en = {c: _english_score(df[c]) for c in df.columns}
    en_col = max(en, key=en.get)
    ur = {c: _urdu_score(df[c]) for c in df.columns if c != en_col}
    ur_col = max(ur, key=ur.get)
    print(f"  columns: {list(df.columns)}")
    print(f"  english scores: { {k: round(v, 2) for k, v in en.items()} }")
    print(f"  urdu scores:    { {k: round(v, 2) for k, v in ur.items()} }")
    print(f"  -> English = '{en_col}', Urdu = '{ur_col}'")
    if en[en_col] < 0.2 or ur[ur_col] < 0.2:
        raise ValueError(f"Could not confidently detect columns in {fname}. "
                         f"Set COLUMN_OVERRIDES['{fname}'] = (english_col, urdu_col).")
    return en_col, ur_col

In [5]:
def load_and_clean(fname, domain):
    print(f"\n=== {fname} ({domain}) ===")
    path = find_file(fname)
    raw = read_csv_robust(path)
    en_col, ur_col = detect_columns(raw, fname)
    df = pd.DataFrame({"src": raw[en_col], "tgt": raw[ur_col]})
    log = {"raw": len(df)}

    df["src"] = df["src"].map(clean_en)
    df["tgt"] = df["tgt"].map(clean_ur)

    df = df[(df.src.str.len() >= 2) & (df.tgt.str.len() >= 2)]
    log["non_empty"] = len(df)

    df = df[df.src.str.lower() != df.tgt.str.lower()]
    log["src!=tgt"] = len(df)

    df = df[df.src.map(lambda x: script_ratio(x, "en")) >= MIN_SCRIPT_RATIO]
    df = df[df.tgt.map(lambda x: script_ratio(x, "ur")) >= MIN_SCRIPT_RATIO]
    log["script_ok"] = len(df)

    ns, nt = df.src.str.split().str.len(), df.tgt.str.split().str.len()
    keep = (ns <= MAX_WORDS) & (nt <= MAX_WORDS)
    r = ns / nt.clip(lower=1)
    bad_ratio = (np.minimum(ns, nt) >= 5) & ((r > WORD_RATIO_RANGE[1]) | (r < WORD_RATIO_RANGE[0]))
    df = df[keep & ~bad_ratio]
    log["len_ratio_ok"] = len(df)

    df = df.drop_duplicates(["src", "tgt"])
    log["dedup_pairs"] = len(df)

    df["domain"] = domain
    print("  filter log:", log)
    return df.reset_index(drop=True)


frames = [load_and_clean(f, d) for f, d in FILES.items()]
data = pd.concat(frames, ignore_index=True)
print("\nCombined before cross-dataset dedup:", len(data))


=== english_urdu.csv (general) ===
  columns: ['english', 'urdu']
  english scores: {'english': 0.71, 'urdu': 0.0}
  urdu scores:    {'urdu': 0.7}
  -> English = 'english', Urdu = 'urdu'
  filter log: {'raw': 1663, 'non_empty': 1663, 'src!=tgt': 1663, 'script_ok': 1662, 'len_ratio_ok': 1662, 'dedup_pairs': 1662}

=== parallel-corpus.csv (parallel) ===
  columns: ['english', 'urdu']
  english scores: {'english': 0.86, 'urdu': 0.0}
  urdu scores:    {'urdu': 0.9}
  -> English = 'english', Urdu = 'urdu'
  filter log: {'raw': 23896, 'non_empty': 23883, 'src!=tgt': 23854, 'script_ok': 23692, 'len_ratio_ok': 23668, 'dedup_pairs': 23627}

=== quran.csv (quran) ===
  columns: ['english', 'urdu']
  english scores: {'english': 0.99, 'urdu': 0.0}
  urdu scores:    {'urdu': 1.0}
  -> English = 'english', Urdu = 'urdu'
  filter log: {'raw': 6366, 'non_empty': 6366, 'src!=tgt': 6366, 'script_ok': 6366, 'len_ratio_ok': 6351, 'dedup_pairs': 6349}

Combined before cross-dataset dedup: 31638


In [6]:
_key = lambda s: re.sub(r"\W+", "", s.lower())
data["_k"] = data.src.map(_key)
data = data.drop_duplicates("_k", keep="first").drop(columns="_k").reset_index(drop=True)
print("After source-level dedup:", len(data))

After source-level dedup: 28186


In [7]:
def labse_filter(df, thr, bs=256):
    from sentence_transformers import SentenceTransformer
    m = SentenceTransformer("sentence-transformers/LaBSE", device="cuda" if USE_CUDA else "cpu")
    a = m.encode(df.src.tolist(), batch_size=bs, normalize_embeddings=True, show_progress_bar=True)
    b = m.encode(df.tgt.tolist(), batch_size=bs, normalize_embeddings=True, show_progress_bar=True)
    sim = (a * b).sum(1)
    print(f"LaBSE: dropping {(sim < thr).sum()} / {len(df)} pairs below {thr}")
    out = df[sim >= thr].reset_index(drop=True)
    del m
    torch.cuda.empty_cache()
    return out


if USE_LABSE_FILTER:
    data = labse_filter(data, LABSE_THRESHOLD)

In [8]:
tok = AutoTokenizer.from_pretrained(MODEL_NAME, src_lang=SRC_LANG, tgt_lang=TGT_LANG)
TGT_ID = tok.convert_tokens_to_ids(TGT_LANG)


def token_lengths(texts, target=False, chunk=50_000):
    lens, unk = [], 0
    for i in range(0, len(texts), chunk):
        part = texts[i:i + chunk]
        ids = tok(text_target=part, truncation=False)["input_ids"] if target else tok(part, truncation=False)["input_ids"]
        lens.extend(len(x) for x in ids)
        if target:
            unk += sum(x.count(tok.unk_token_id) for x in ids)
    return np.array(lens), unk


ls, _ = token_lengths(data.src.tolist())
lt, unk = token_lengths(data.tgt.tolist(), target=True)
print(f"Token length  src: mean={ls.mean():.1f} p95={np.percentile(ls, 95):.0f} | "
      f"tgt: mean={lt.mean():.1f} p95={np.percentile(lt, 95):.0f}")
print(f"<unk> rate on Urdu targets: {unk / lt.sum():.5%}")
data = data[(ls <= MAX_LEN) & (lt <= MAX_LEN)].reset_index(drop=True)
print("After token-length filter:", len(data))
print(data.domain.value_counts().to_string())

HTTP Request: HEAD https://huggingface.co/facebook/nllb-200-distilled-600M/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/nllb-200-distilled-600M/f8d333a098d19b4fd9a8b18f94170487ad3f821d/config.json "HTTP/1.1 200 OK"
HTTP Request: GET https://huggingface.co/api/resolve-cache/models/facebook/nllb-200-distilled-600M/f8d333a098d19b4fd9a8b18f94170487ad3f821d/config.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

HTTP Request: HEAD https://huggingface.co/facebook/nllb-200-distilled-600M/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/nllb-200-distilled-600M/f8d333a098d19b4fd9a8b18f94170487ad3f821d/tokenizer_config.json "HTTP/1.1 200 OK"
HTTP Request: GET https://huggingface.co/api/resolve-cache/models/facebook/nllb-200-distilled-600M/f8d333a098d19b4fd9a8b18f94170487ad3f821d/tokenizer_config.json "HTTP/1.1 200 OK"


tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

HTTP Request: HEAD https://huggingface.co/facebook/nllb-200-distilled-600M/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/nllb-200-distilled-600M/f8d333a098d19b4fd9a8b18f94170487ad3f821d/tokenizer_config.json "HTTP/1.1 200 OK"
HTTP Request: GET https://huggingface.co/api/models/facebook/nllb-200-distilled-600M/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
HTTP Request: GET https://huggingface.co/api/models/facebook/nllb-200-distilled-600M/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/facebook/nllb-200-distilled-600M/resolve/main/sentencepiece.bpe.model "HTTP/1.1 302 Found"
HTTP Request: GET https://huggingface.co/api/models/facebook/nllb-200-distilled-600M/xet-read-token/f8d333a098d19b4fd9a8b18f94170487ad3f821d "HTTP/1.1 200 OK"


sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 4.85MB            

... (4 outputs left out)

HTTP Request: HEAD https://huggingface.co/facebook/nllb-200-distilled-600M/resolve/main/added_tokens.json "HTTP/1.1 404 Not Found"
HTTP Request: HEAD https://huggingface.co/facebook/nllb-200-distilled-600M/resolve/main/special_tokens_map.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/nllb-200-distilled-600M/f8d333a098d19b4fd9a8b18f94170487ad3f821d/special_tokens_map.json "HTTP/1.1 200 OK"
HTTP Request: GET https://huggingface.co/api/resolve-cache/models/facebook/nllb-200-distilled-600M/f8d333a098d19b4fd9a8b18f94170487ad3f821d/special_tokens_map.json "HTTP/1.1 200 OK"


special_tokens_map.json:   0%|          | 0.00/3.55k [00:00<?, ?B/s]

HTTP Request: HEAD https://huggingface.co/facebook/nllb-200-distilled-600M/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"
HTTP Request: GET https://huggingface.co/api/models/facebook/nllb-200-distilled-600M "HTTP/1.1 200 OK"


Token length  src: mean=27.7 p95=81 | tgt: mean=29.5 p95=84
<unk> rate on Urdu targets: 0.06561%
After token-length filter: 28141
domain
parallel    20470
quran        6301
general      1370


In [9]:
def strat_split(df, test_size):
    try:
        return train_test_split(df, test_size=test_size, stratify=df.domain, random_state=SEED)
    except ValueError:
        return train_test_split(df, test_size=test_size, random_state=SEED)


n_test = min(TEST_MAX, int(len(data) * 0.05))
n_val = min(VAL_MAX, int(len(data) * 0.05))
rest, test = strat_split(data, n_test)
train, val = strat_split(rest, n_val)
if MAX_TRAIN_ROWS:
    train = train.sample(min(MAX_TRAIN_ROWS, len(train)), random_state=SEED)
train, val, test = [d.reset_index(drop=True) for d in (train, val, test)]
print(f"train={len(train)} val={len(val)} test={len(test)}")

train.to_csv(f"{OUT_DIR}/train_clean.csv", index=False)
val.to_csv(f"{OUT_DIR}/val_clean.csv", index=False)
test.to_csv(f"{OUT_DIR}/test_clean.csv", index=False)

train=25327 val=1407 test=1407


In [10]:
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model.generation_config.forced_bos_token_id = TGT_ID   # generation settings live ONLY in generation_config
if USE_CUDA:
    model.to("cuda")
if FREEZE_EMBEDDINGS:
    for p in model.get_input_embeddings().parameters():
        p.requires_grad = False
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params: {n_train / 1e6:.0f}M of {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M")

HTTP Request: HEAD https://huggingface.co/facebook/nllb-200-distilled-600M/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/nllb-200-distilled-600M/f8d333a098d19b4fd9a8b18f94170487ad3f821d/config.json "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/facebook/nllb-200-distilled-600M/resolve/main/adapter_config.json "HTTP/1.1 404 Not Found"
HTTP Request: HEAD https://huggingface.co/facebook/nllb-200-distilled-600M/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/nllb-200-distilled-600M/f8d333a098d19b4fd9a8b18f94170487ad3f821d/config.json "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/facebook/nllb-200-distilled-600M/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
HTTP Request: HEAD https://huggingface.co/facebook/nllb-200-distilled-600M/resolve/main/model.safetensors.index.json "HTTP/1.1

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 2.46GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

HTTP Request: HEAD https://huggingface.co/facebook/nllb-200-distilled-600M/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
HTTP Request: GET https://huggingface.co/api/models/facebook/nllb-200-distilled-600M "HTTP/1.1 200 OK"
HTTP Request: GET https://huggingface.co/api/models/facebook/nllb-200-distilled-600M/commits/main "HTTP/1.1 200 OK"
HTTP Request: GET https://huggingface.co/api/models/facebook/nllb-200-distilled-600M/discussions?p=0 "HTTP/1.1 200 OK"


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

HTTP Request: GET https://huggingface.co/api/models/facebook/nllb-200-distilled-600M/commits/refs%2Fpr%2F45 "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/facebook/nllb-200-distilled-600M/resolve/refs%2Fpr%2F45/model.safetensors.index.json "HTTP/1.1 404 Not Found"
HTTP Request: HEAD https://huggingface.co/facebook/nllb-200-distilled-600M/resolve/refs%2Fpr%2F45/model.safetensors "HTTP/1.1 302 Found"
HTTP Request: HEAD https://huggingface.co/facebook/nllb-200-distilled-600M/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/nllb-200-distilled-600M/f8d333a098d19b4fd9a8b18f94170487ad3f821d/generation_config.json "HTTP/1.1 200 OK"
HTTP Request: GET https://huggingface.co/api/resolve-cache/models/facebook/nllb-200-distilled-600M/f8d333a098d19b4fd9a8b18f94170487ad3f821d/generation_config.json "HTTP/1.1 200 OK"


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

HTTP Request: GET https://huggingface.co/api/models/facebook/nllb-200-distilled-600M/xet-read-token/a3e77be725cf30383f1faeb8d2f0b0ea98ac554e "HTTP/1.1 200 OK"


model.safetensors: reconstructing file:   0%|          |  0.00B / 2.46GB            

model.safetensors: downloading bytes:           |  0.00B            

Trainable params: 353M of 615M


In [11]:
@torch.no_grad()
def translate(texts, batch_size=GEN_BATCH, num_beams=NUM_BEAMS, max_new_tokens=MAX_LEN + 20):
    model.eval()
    order = np.argsort([-len(t) for t in texts])     # longest first: fewer padding tokens, OOM shows up early
    out = [None] * len(texts)
    for i in range(0, len(texts), batch_size):
        idx = order[i:i + batch_size]
        batch = [texts[j] for j in idx]
        enc = tok(batch, return_tensors="pt", padding=True, truncation=True, max_length=MAX_LEN).to(model.device)
        with torch.autocast("cuda", dtype=torch.float16, enabled=USE_CUDA):
            gen = model.generate(**enc, forced_bos_token_id=TGT_ID, num_beams=num_beams,
                                 max_new_tokens=max_new_tokens)
        for j, d in zip(idx, tok.batch_decode(gen, skip_special_tokens=True)):
            out[j] = d
    return out


_chrf = CHRF(word_order=2)   # chrF++
_bleu = BLEU()


def corpus_scores(h, r):
    return {"BLEU": _bleu.corpus_score(h, [r]).score, "chrF++": _chrf.corpus_score(h, [r]).score}


# ---- METEOR (NLTK). NLTK only ships English WordNet and an English stemmer, so for Urdu we switch both
# off: this is METEOR with exact-word matching + the usual fragmentation penalty. It still works offline
# (no nltk.download needed). Report it as "METEOR (exact match)" in a paper.
from nltk.translate.meteor_score import meteor_score as _nltk_meteor


class _NoWordNet:
    def synsets(self, *a, **k):
        return []


class _NoStemmer:
    def stem(self, w):
        return w


_NO_WN, _NO_STEM = _NoWordNet(), _NoStemmer()


def sentence_meteor_scores(hyps, refs):
    """Sentence-level METEOR in 0-100. Inputs must already be clean_ur()-normalised."""
    out = []
    for h, r in zip(hyps, refs):
        ht, rt = h.split(), r.split()
        if not ht or not rt:
            out.append(0.0)
            continue
        out.append(100.0 * _nltk_meteor([rt], ht, preprocess=lambda x: x, stemmer=_NO_STEM, wordnet=_NO_WN))
    return np.array(out)


def corpus_meteor(h, r):
    return float(sentence_meteor_scores(h, r).mean())


# ---- COMET (neural, uses the source sentence too)
def compute_comet(srcs, systems, refs):
    """Segment-level COMET for every system in `systems` ({name: hypotheses}). Returns {name: np.array}.
    NLLB is parked on the CPU while COMET (XLM-R large) uses the GPU, then moved back."""
    try:
        from comet import download_model, load_from_checkpoint
    except Exception as e:  # noqa
        print("COMET is not installed -> skipped:", e)
        return {}
    comet_model, scores = None, {}
    try:
        model.to("cpu")
        gc.collect()
        if USE_CUDA:
            torch.cuda.empty_cache()
        comet_model = load_from_checkpoint(download_model(COMET_MODEL))
        for name, hyps in systems.items():
            data_c = [{"src": s, "mt": clean_ur(h), "ref": r} for s, h, r in zip(srcs, hyps, refs)]
            out = comet_model.predict(data_c, batch_size=COMET_BS, gpus=1 if USE_CUDA else 0, progress_bar=True)
            scores[name] = np.asarray(out.scores, dtype=float)
            print(f"COMET [{name}]: {scores[name].mean():.4f}")
    except Exception as e:  # noqa
        print("COMET failed -> skipped (everything else still runs):", repr(e))
    finally:
        del comet_model
        gc.collect()
        if USE_CUDA:
            torch.cuda.empty_cache()
            model.to("cuda")
    return scores


def evaluate_system(name, hyps, refs, domains, comet=None):
    """BLEU, chrF++ (+ METEOR, + COMET if given) overall and per domain. `comet` = segment scores or None."""
    h = [clean_ur(x) for x in hyps]      # same normalisation on both sides, so letter variants don't penalise
    r = [clean_ur(x) for x in refs]
    met = sentence_meteor_scores(h, r) if RUN_METEOR else None

    def row(subset, idx):
        d = {"system": name, "subset": subset, "n": len(idx),
             **corpus_scores([h[i] for i in idx], [r[i] for i in idx])}
        if met is not None:
            d["METEOR"] = float(met[idx].mean())
        if comet is not None:
            d["COMET"] = float(np.asarray(comet)[idx].mean())
        return d

    rows = [row("ALL", np.arange(len(h)))]
    for dom in sorted(set(domains)):
        rows.append(row(dom, np.array([i for i, x in enumerate(domains) if x == dom])))
    return pd.DataFrame(rows)


def sentence_chrf(hyps, refs):
    return np.array([_chrf.sentence_score(clean_ur(h), [clean_ur(r)]).score for h, r in zip(hyps, refs)])


def paired_bootstrap(a, b, n=1000, seed=0):
    """Is system B better than A? Uses sentence-level chrF++ scores."""
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(a), (n, len(a)))
    diffs = (b[idx] - a[idx]).mean(1)
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    return diffs.mean(), (lo, hi), float((diffs <= 0).mean())


def has_repetition(t, n=3, k=3):
    w = t.split()
    if len(w) < n * k:
        return False
    return max(Counter(tuple(w[i:i + n]) for i in range(len(w) - n + 1)).values()) >= k


def error_analysis(df, col, out_path):
    h = df[col].fillna("").tolist()
    ref_len = df.tgt.str.split().str.len().clip(lower=1)
    hyp_len = pd.Series([len(x.split()) for x in h])
    stats = {
        "empty_outputs": int(sum(1 for x in h if not x.strip())),
        "hyp/ref length ratio (mean)": float((hyp_len / ref_len).mean()),
        "outputs w/ >20% Latin letters (untranslated)": int(sum(script_ratio(x, "en") > 0.2 for x in h if x.strip())),
        "outputs w/ repeated 3-grams (degeneration)": int(sum(has_repetition(x) for x in h)),
    }
    print(f"\nError analysis [{col}]:")
    for k, v in stats.items():
        print(f"  {k}: {v}")
    # length-bucket scores
    buckets = pd.cut(df.src.str.split().str.len(), [0, 5, 10, 20, 40, 1000], labels=["1-5", "6-10", "11-20", "21-40", "40+"])
    print("  sentence chrF++ by source length:")
    print(df.groupby(buckets)[f"chrf_{col}"].mean().round(1).to_string())
    df.sort_values(f"chrf_{col}").head(50)[["src", "tgt", col, f"chrf_{col}", "domain"]].to_csv(out_path, index=False)
    print(f"  50 worst examples -> {out_path}")

In [12]:
results = []
test_out = test.copy()
if RUN_BASELINE:
    print("\nTranslating test set with zero-shot NLLB ...")
    test_out["baseline"] = translate(test.src.tolist())
    results.append(evaluate_system("zero-shot NLLB", test_out.baseline.tolist(), test.tgt.tolist(), test.domain.tolist()))
    print(results[-1].round(2).to_string(index=False))


Translating test set with zero-shot NLLB ...


[transformers] Both `max_new_tokens` (=220) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=220) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=220) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=220) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

        system   subset    n  BLEU  chrF++  METEOR
zero-shot NLLB      ALL 1407 26.31   48.19   49.39
zero-shot NLLB  general   69 15.62   41.57   38.35
zero-shot NLLB parallel 1023 29.44   51.16   51.89
zero-shot NLLB    quran  315 20.67   42.80   43.70


In [13]:
def tokenize_batch(b):
    return tok(b["src"], text_target=b["tgt"], max_length=MAX_LEN, truncation=True)


def to_hf(df):
    ds = Dataset.from_pandas(df[["src", "tgt"]].reset_index(drop=True))
    return ds.map(tokenize_batch, batched=True, remove_columns=["src", "tgt"], desc="tokenising")


train_ds = to_hf(train)
val_ds = to_hf(val.sample(min(len(val), VAL_EVAL_SUBSET), random_state=SEED))
from transformers.models.m2m_100.modeling_m2m_100 import shift_tokens_right


class NLLBCollator(DataCollatorForSeq2Seq):
    """NLLB (M2M100) has no prepare_decoder_input_ids_from_labels(), and label smoothing removes `labels`
    before the forward pass, so the decoder would receive no inputs. Build decoder_input_ids ourselves."""

    def __call__(self, features, return_tensors=None):
        batch = super().__call__(features, return_tensors)
        batch["decoder_input_ids"] = shift_tokens_right(
            batch["labels"], tok.pad_token_id, model.config.decoder_start_token_id)
        return batch


collator = NLLBCollator(tok, model=model, pad_to_multiple_of=8, label_pad_token_id=-100)


def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    preds = np.where(preds != -100, preds, tok.pad_token_id)
    labels = np.where(labels != -100, labels, tok.pad_token_id)
    h = [clean_ur(x) for x in tok.batch_decode(preds, skip_special_tokens=True)]
    r = [clean_ur(x) for x in tok.batch_decode(labels, skip_special_tokens=True)]
    s = corpus_scores(h, r)
    out = {"chrf": s["chrF++"], "bleu": s["BLEU"], "gen_len": float(np.mean([len(x.split()) for x in h]))}
    if RUN_METEOR:
        out["meteor"] = corpus_meteor(h, r)
    return out


ta_params = inspect.signature(Seq2SeqTrainingArguments.__init__).parameters
steps_per_epoch = -(-len(train_ds) // (TRAIN_BS * GRAD_ACC))
warmup_steps = max(1, int(0.05 * steps_per_epoch * EPOCHS))      # 5% warmup, as an int (works in every version)
print(f"steps/epoch={steps_per_epoch}, total={steps_per_epoch * EPOCHS}, warmup={warmup_steps}")
args_kw = dict(
    output_dir=f"{OUT_DIR}/checkpoints",
    per_device_train_batch_size=TRAIN_BS, per_device_eval_batch_size=EVAL_BS,
    gradient_accumulation_steps=GRAD_ACC,
    learning_rate=LR, num_train_epochs=EPOCHS, warmup_steps=warmup_steps, weight_decay=0.01,
    lr_scheduler_type="linear", label_smoothing_factor=LABEL_SMOOTHING,
    optim=OPTIM, fp16=USE_CUDA,
    group_by_length=True,                          # older transformers
    train_sampling_strategy="group_by_length",     # newer transformers (unsupported keys are dropped below)
    predict_with_generate=True, generation_max_length=MAX_LEN, generation_num_beams=1,
    save_strategy="epoch", logging_steps=100,
    load_best_model_at_end=True, metric_for_best_model="chrf", greater_is_better=True,
    save_total_limit=2, report_to="none", seed=SEED, dataloader_num_workers=2,
)
args_kw["eval_strategy" if "eval_strategy" in ta_params else "evaluation_strategy"] = "epoch"
_ignored = [k for k in args_kw if k not in ta_params]
if _ignored:
    print("Ignored (not supported by this transformers version):", _ignored)
args = Seq2SeqTrainingArguments(**{k: v for k, v in args_kw.items() if k in ta_params})

tr_kw = dict(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
             data_collator=collator, compute_metrics=compute_metrics,
             callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])
tr_params = inspect.signature(Seq2SeqTrainer.__init__).parameters
tr_kw["processing_class" if "processing_class" in tr_params else "tokenizer"] = tok
trainer = Seq2SeqTrainer(**tr_kw)

tokenising:   0%|          | 0/25327 [00:00<?, ? examples/s]

tokenising:   0%|          | 0/800 [00:00<?, ? examples/s]

steps/epoch=1583, total=7915, warmup=395
Ignored (not supported by this transformers version): ['group_by_length']


In [14]:
trainer.train(resume_from_checkpoint=RESUME_FROM)
final_dir = f"{OUT_DIR}/final_model"
trainer.save_model(final_dir)
tok.save_pretrained(final_dir)
pd.DataFrame(trainer.state.log_history).to_csv(f"{OUT_DIR}/training_log.csv", index=False)
print("Saved model to", final_dir)

Epoch,Training Loss,Validation Loss,Chrf,Bleu,Gen Len,Meteor
1,7.355834,0.810024,56.841470,37.760696,20.475000,63.923577
2,5.751298,0.755261,60.996382,41.870533,21.680000,68.102649
3,4.846438,0.745691,62.081259,43.291437,21.753750,69.369270
4,4.120144,0.755090,62.048456,43.485160,21.686250,69.197551
5,3.732025,0.759321,62.436023,43.967342,21.821250,69.807031


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved model to /kaggle/working/nllb_en_ur/final_model


In [15]:
print("\nTranslating test set with fine-tuned model ...")
test_out["finetuned"] = translate(test.src.tolist())

systems = {"baseline": test_out.baseline.tolist()} if RUN_BASELINE else {}
systems["finetuned"] = test_out.finetuned.tolist()

# COMET runs here so that it lands in the same table as BLEU / chrF++ / METEOR
comet_scores = compute_comet(test.src.tolist(), systems, test.tgt.tolist()) if RUN_COMET else {}
for _n, _sc in comet_scores.items():
    test_out[f"comet_{_n}"] = _sc

refs_l, doms_l = test.tgt.tolist(), test.domain.tolist()
parts = []
if RUN_BASELINE:
    parts.append(evaluate_system("zero-shot NLLB", systems["baseline"], refs_l, doms_l, comet_scores.get("baseline")))
parts.append(evaluate_system("fine-tuned", systems["finetuned"], refs_l, doms_l, comet_scores.get("finetuned")))
report = pd.concat(parts, ignore_index=True)
print("\n=== TEST RESULTS ===")
print(report.round(2).to_string(index=False))
report.to_csv(f"{OUT_DIR}/test_metrics.csv", index=False)

# significance + error analysis
test_out["chrf_finetuned"] = sentence_chrf(test_out.finetuned, test.tgt)
if RUN_BASELINE:
    test_out["chrf_baseline"] = sentence_chrf(test_out.baseline, test.tgt)
    d, (lo, hi), p = paired_bootstrap(test_out.chrf_baseline.values, test_out.chrf_finetuned.values)
    print(f"\nPaired bootstrap (fine-tuned - baseline, sentence chrF++): mean diff={d:.2f}, "
          f"95% CI=[{lo:.2f}, {hi:.2f}], p~{p:.4f}")
    error_analysis(test_out, "baseline", f"{OUT_DIR}/worst_baseline.csv")
error_analysis(test_out, "finetuned", f"{OUT_DIR}/worst_finetuned.csv")
test_out.to_csv(f"{OUT_DIR}/test_predictions.csv", index=False)

[transformers] Both `max_new_tokens` (=220) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Translating test set with fine-tuned model ...


[transformers] Both `max_new_tokens` (=220) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=220) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=220) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=220) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface


=== TEST RESULTS ===
        system   subset    n  BLEU  chrF++  METEOR
zero-shot NLLB      ALL 1407 26.31   48.19   49.39
zero-shot NLLB  general   69 15.62   41.57   38.35
zero-shot NLLB parallel 1023 29.44   51.16   51.89
zero-shot NLLB    quran  315 20.67   42.80   43.70
    fine-tuned      ALL 1407 45.08   63.37   68.96
    fine-tuned  general   69 21.35   46.84   48.09
    fine-tuned parallel 1023 48.88   66.26   71.82
    fine-tuned    quran  315 39.38   58.85   64.26

Paired bootstrap (fine-tuned - baseline, sentence chrF++): mean diff=16.70, 95% CI=[15.67, 17.73], p~0.0000

Error analysis [baseline]:
  empty_outputs: 0
  hyp/ref length ratio (mean): 0.9710326420490073
  outputs w/ >20% Latin letters (untranslated): 85
  outputs w/ repeated 3-grams (degeneration): 7
  sentence chrF++ by source length:
src
1-5      55.7
6-10     53.0
11-20    48.0
21-40    47.5
40+      45.2
  50 worst examples -> /kaggle/working/nllb_en_ur/worst_baseline.csv

Error analysis [finetuned]:
  empt

In [16]:
def translate_en_to_ur(sentences, beams=NUM_BEAMS):
    if isinstance(sentences, str):
        sentences = [sentences]
    return translate([clean_en(s) for s in sentences], num_beams=beams)


for s in ["How are you today?", "Knowledge is better than wealth.", "The meeting has been postponed until Monday."]:
    print(s, "->", translate_en_to_ur(s)[0])

[transformers] Both `max_new_tokens` (=220) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=220) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=220) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


How are you today? -> آپ کیسے ہیں آج؟
Knowledge is better than wealth. -> علم دولت سے بہتر ہے۔
The meeting has been postponed until Monday. -> اجلاس پیر تک ملتوی کر دیا گیا ہے۔


In [18]:
import zipfile
from IPython.display import FileLink, display

# overall numbers in one small JSON file
summary = {"model": MODEL_NAME, "test_rows": int(len(test)), "results": report.round(3).to_dict(orient="records")}
with open(f"{OUT_DIR}/final_metrics.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

metric_files = ["final_metrics.json", "test_metrics.csv", "training_log.csv",
                "test_predictions.csv", "worst_baseline.csv", "worst_finetuned.csv"]
metrics_zip = os.path.join(os.path.dirname(OUT_DIR.rstrip("/")) or ".", "nllb_en_ur_metrics.zip")
with zipfile.ZipFile(metrics_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for fn in metric_files:
        fp = os.path.join(OUT_DIR, fn)
        if os.path.exists(fp):
            z.write(fp, arcname=fn)
print("Metrics zip:", metrics_zip, f"({os.path.getsize(metrics_zip) / 1e6:.1f} MB)")

# clickable links (work in an interactive Kaggle/Jupyter session)
os.chdir(os.path.dirname(metrics_zip))
display(FileLink(os.path.basename(metrics_zip)))
model_zip = os.path.join(os.path.dirname(metrics_zip), "nllb_en_ur_finetuned_fp16.zip")
if os.path.exists(model_zip):
    display(FileLink(os.path.basename(model_zip)))

Metrics zip: /kaggle/working/nllb_en_ur_metrics.zip (0.3 MB)


/kaggle/working/nllb_en_ur_metrics.zip

In [20]:
# Contains: fp16 weights + model config + language codes. Needs ~1.2 GB. Use predict_local.py to run it.
PICKLE_PATH = f"{OUT_DIR}/nllb_en_ur_model.pkl"
_state = {k: v.detach().to("cpu", dtype=torch.float16) for k, v in model.state_dict().items()}
torch.save({
    "state_dict": _state,
    "config": model.config.to_dict(),
    "base_model": MODEL_NAME,
    "src_lang": SRC_LANG, "tgt_lang": TGT_LANG,
    "max_len": MAX_LEN,
}, PICKLE_PATH)
del _state
print("Pickle saved:", PICKLE_PATH, f"({os.path.getsize(PICKLE_PATH) / 1e9:.2f} GB)")
display(FileLink(os.path.relpath(PICKLE_PATH, os.path.dirname(OUT_DIR.rstrip('/')) or ".")))

Pickle saved: /kaggle/working/nllb_en_ur/nllb_en_ur_model.pkl (2.80 GB)


/kaggle/working/nllb_en_ur/nllb_en_ur_model.pkl

In [21]:
# ===== COMET on the saved predictions (standalone cell; run after training) =====
import os, sys, gc, subprocess, importlib, importlib.util, traceback
import numpy as np, pandas as pd, torch
from importlib.metadata import version, PackageNotFoundError

OUT_DIR = "/kaggle/working/nllb_en_ur"
COMET_MODEL = "Unbabel/wmt22-comet-da"      # lighter option: "Unbabel/eamt22-cometinho-da"
COMET_BS = 16                               # use 8 if the GPU runs out of memory


def run_comet():
    # 1) free the GPU if this notebook still holds the NLLB model
    if "model" in globals():
        try:
            globals()["model"].to("cpu")
        except Exception:
            pass
    gc.collect()
    torch.cuda.empty_cache()

    # 2) install COMET safely (key packages frozen). Needed again if the session was restarted.
    if importlib.util.find_spec("comet") is None:
        pins = []
        for n in ["numpy", "pandas", "pyarrow", "scipy", "scikit-learn", "torch", "transformers", "tokenizers",
                  "datasets", "huggingface-hub", "protobuf", "sentencepiece", "safetensors"]:
            try:
                pins.append(f"{n}=={version(n)}")
            except PackageNotFoundError:
                pass
        with open("/tmp/constraints.txt", "w") as f:
            f.write("\n".join(pins) + "\n")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "unbabel-comet"])
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-c", "/tmp/constraints.txt",
                               "entmax", "jsonargparse", "pytorch-lightning", "torchmetrics"])
        importlib.invalidate_caches()
    from comet import download_model, load_from_checkpoint

    # 3) load + fix: COMET 2.2.7 expects the encoder to return 3 values, newer transformers return 2
    #    (that was the "expected 3, got 2" error).
    cm = load_from_checkpoint(download_model(COMET_MODEL))
    hf = cm.encoder.model
    orig = hf.forward

    def fwd(*a, **k):
        out = orig(*a, **k)
        return (out[0], None, out[1]) if isinstance(out, tuple) and len(out) == 2 else out

    hf.forward = fwd

    # 4) score every system found in test_predictions.csv
    df = pd.read_csv(f"{OUT_DIR}/test_predictions.csv")
    clean = globals().get("clean_ur", lambda x: x)      # same Urdu normalisation as the other metrics, if available
    systems = [c for c in ("baseline", "finetuned") if c in df.columns]
    for name in systems:
        data = [{"src": str(s), "mt": clean(str(h)), "ref": str(r)}
                for s, h, r in zip(df.src, df[name].fillna(""), df.tgt)]
        out = cm.predict(data, batch_size=COMET_BS, gpus=1 if torch.cuda.is_available() else 0, progress_bar=True)
        df[f"comet_{name}"] = np.asarray(out.scores, dtype=float)
        print(f"COMET [{name}] = {df[f'comet_{name}'].mean():.4f}")

    # 5) summary overall + per domain, and merge into the metrics table
    rows = []
    for name in systems:
        col = f"comet_{name}"
        rows.append({"system": name, "subset": "ALL", "n": len(df), "COMET": df[col].mean()})
        for d, g in df.groupby("domain"):
            rows.append({"system": name, "subset": d, "n": len(g), "COMET": g[col].mean()})
    summary = pd.DataFrame(rows)
    summary.to_csv(f"{OUT_DIR}/comet_summary.csv", index=False)
    df.to_csv(f"{OUT_DIR}/test_predictions_with_comet.csv", index=False)
    print(summary.round(4).to_string(index=False))

    mp = f"{OUT_DIR}/test_metrics.csv"
    if os.path.exists(mp):
        rep = pd.read_csv(mp).drop(columns=["COMET"], errors="ignore")
        summary["system"] = summary["system"].map({"baseline": "zero-shot NLLB", "finetuned": "fine-tuned"})
        rep = rep.merge(summary[["system", "subset", "COMET"]], on=["system", "subset"], how="left")
        rep.to_csv(f"{OUT_DIR}/test_metrics_with_comet.csv", index=False)
        print("\n=== TEST RESULTS (with COMET) ===")
        print(rep.round(2).to_string(index=False))


try:
    run_comet()
except Exception:
    traceback.print_exc()
    print("\nCOMET failed. Copy the whole traceback above and send it to me.")

HTTP Request: GET https://huggingface.co/api/models/Unbabel/wmt22-comet-da/revision/main "HTTP/1.1 200 OK"


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.6. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../root/.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
HTTP Request: HEAD https://huggingface.co/xlm-roberta-large/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
HTTP Request: GET https://huggingface.co/api/models/xlm-roberta-large/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
HTTP Request: GET https://huggingface.co/api/models/FacebookAI/xlm-roberta-large/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
HTTP Request: GET https://huggingface.co/api/models/xlm-roberta-large/tree/main?recursive=true&expand=false "HTTP/1.1 307 Temporary Redirect"
HTTP Request: GET https://huggingface.co/api/models/FacebookAI/xlm-ro

COMET [baseline] = 0.8064


Predicting DataLoader 0: 100%|██████████| 88/88 [00:36<00:00,  2.44it/s]


COMET [finetuned] = 0.8715
   system   subset    n  COMET
 baseline      ALL 1407 0.8064
 baseline  general   69 0.8530
 baseline parallel 1023 0.8125
 baseline    quran  315 0.7764
finetuned      ALL 1407 0.8715
finetuned  general   69 0.8787
finetuned parallel 1023 0.8780
finetuned    quran  315 0.8486

=== TEST RESULTS (with COMET) ===
        system   subset    n  BLEU  chrF++  METEOR  COMET
zero-shot NLLB      ALL 1407 26.31   48.19   49.39   0.81
zero-shot NLLB  general   69 15.62   41.57   38.35   0.85
zero-shot NLLB parallel 1023 29.44   51.16   51.89   0.81
zero-shot NLLB    quran  315 20.67   42.80   43.70   0.78
    fine-tuned      ALL 1407 45.08   63.37   68.96   0.87
    fine-tuned  general   69 21.35   46.84   48.09   0.88
    fine-tuned parallel 1023 48.88   66.26   71.82   0.88
    fine-tuned    quran  315 39.38   58.85   64.26   0.85


In [22]:
# ===== Download the tuned model from Kaggle =====
import os, zipfile
from IPython.display import FileLink, display

WHAT = "fp16"        # "fp16" = recommended (~1.2 GB) | "full" = full precision (~2.4 GB) | "pickle" = the .pkl file
WORK = "/kaggle/working"
OUT_DIR = f"{WORK}/nllb_en_ur"


def zip_folder(folder, zip_path):
    """Zip without compression (model weights do not compress, so this is much faster)."""
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_STORED) as z:
        for root, _, files in os.walk(folder):
            for fn in files:
                full = os.path.join(root, fn)
                z.write(full, os.path.relpath(full, folder))


if WHAT == "pickle":
    target = f"{OUT_DIR}/nllb_en_ur_model.pkl"
    if not os.path.exists(target):
        raise FileNotFoundError("nllb_en_ur_model.pkl not found - run step 20 of the script first.")
else:
    folder = f"{OUT_DIR}/final_model_fp16" if WHAT == "fp16" else f"{OUT_DIR}/final_model"
    if not os.path.isdir(folder):
        if WHAT == "fp16" and "model" in globals() and "tok" in globals():      # model still in memory -> save it now
            globals()["model"].to("cpu").half().save_pretrained(folder, safe_serialization=True)
            globals()["tok"].save_pretrained(folder)
        else:
            raise FileNotFoundError(f"{folder} not found. Existing: {os.listdir(OUT_DIR) if os.path.isdir(OUT_DIR) else 'nothing'}")
    target = f"{WORK}/nllb_en_ur_finetuned_{WHAT}.zip"
    print("zipping", folder, "...")
    zip_folder(folder, target)

print(f"Ready: {target}  ({os.path.getsize(target) / 1e9:.2f} GB)")
os.chdir(WORK)
display(FileLink(os.path.relpath(target, WORK)))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

zipping /kaggle/working/nllb_en_ur/final_model_fp16 ...
Ready: /kaggle/working/nllb_en_ur_finetuned_fp16.zip  (1.26 GB)


/kaggle/working/nllb_en_ur_finetuned_fp16.zip